# Cell-of-origin correlation analysis: healthy vs cancer

For the `healthy` and `cancer` samples in the pipeline's `results/` folder:

1. loads the per-gene FFT WPS matrix (`results/<sample>/proj/fft_summaries/fft_<sample>_WPS.tsv.gz`, genes x periods 120-280 bp)
2. scores each gene as the mean FFT intensity over 193-199 bp (same as `collapse_fft_wps.py`)
3. correlates the scores with the Tabula Sapiens reference using several methods and compares them visually
4. plots correlation-vs-period curves ("S curves") per sample and healthy vs cancer

**Colour code:** red = blood/immune groups (Tabula Sapiens immune compartment), grey = all other groups,
black = erythroid groups (erythrocytes and erythroid progenitors). The compartment figures at the end use the four
Tabula Sapiens compartments instead. Erythroid groups are kept out of the blood/immune set: mature erythrocytes have no nucleus and cannot release cfDNA, so they act as a negative control and should
rank near the bottom / stay flat.

## Settings

In [ ]:
from pathlib import Path

# the pipeline's workflow/ directory; ".." works when this notebook sits in workflow/result_analysis/
WORKFLOW_DIR = Path("..")

REF_CSV = WORKFLOW_DIR / "resources/avg_expr_by_celltype_tissue.csv"
# Tabula Sapiens compartment per group (immune / epithelial / endothelial / stromal)
COMPARTMENT_CSV = WORKFLOW_DIR / "resources/celltype_compartment.csv"
RESULTS_DIR = WORKFLOW_DIR / "results"
SAMPLES = ["healthy", "cancer"]

# where plots and the two summary tables are written
OUT_DIR = Path("output")

BAND = (193, 199)                               # period band (bp) used for the gene score
VAR_GENE_COUNTS = [5000, 3000, 2000, 1000, 500] # Pearson on top-N variable genes
CURVE_METHODS = ["baseline", "pearson_var5k"]  # methods drawn as S curves (one panel each)
TOP_N = 20
RANK_METHODS = ["baseline"]   # methods used for the report ranking figures, e.g. ["baseline", "pearson_var5k"]
RANK_TOP = 20                 # bars per top / bottom ranking panel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 45)
pd.set_option("display.width", 250)

def fft_path(sample):
    d = RESULTS_DIR / sample / "proj" / "fft_summaries"
    for name in (f"fft_{sample}_WPS.tsv.gz", f"fft_{sample}_WPS.tsv"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"no FFT matrix for '{sample}' in {d}")

assert REF_CSV.exists(), f"reference not found: {REF_CSV}"
FFT_FILES = {s: fft_path(s) for s in SAMPLES}
for s, p in FFT_FILES.items():
    print(f"{s}: {p}")
print("output:", OUT_DIR.resolve())

# one colour per category and per sample, used by every plot below
CAT_COLORS = {"blood/immune": "tab:red", "other": "tab:gray", "erythroid": "black"}
SAMPLE_COLORS = {"healthy": "tab:blue", "cancer": "tab:orange"}

## Load data

In [ ]:
ref = pd.read_csv(REF_CSV, index_col=0)
gene_names = ref.pop("gene_name")
print(f"reference: {ref.shape[0]} genes x {ref.shape[1]} groups")

comp = pd.read_csv(COMPARTMENT_CSV, index_col=0)["compartment"].reindex(ref.columns)
assert comp.notna().all(), f"groups without compartment: {list(comp.index[comp.isna()])}"

# plotting category: the immune compartment minus erythroid groups (negative control, no nucleus) = blood/immune
def category(group):
    if group.startswith(("erythrocyte", "erythroid")):
        return "erythroid"
    return "blood/immune" if comp[group] == "immune" else "other"

cat = pd.Series({g: category(g) for g in ref.columns})
cat_color = cat.map(CAT_COLORS)
print(cat.value_counts().to_string())
print(comp.value_counts().to_string())

def load_fft(path):
    fft = pd.read_csv(path, sep="\t", index_col=0)
    fft.columns = fft.columns.astype(float)
    return fft[sorted(fft.columns)]

fft = {s: load_fft(p) for s, p in FFT_FILES.items()}
scores = {}
for s, m in fft.items():
    band_cols = [c for c in m.columns if BAND[0] <= c <= BAND[1]]
    scores[s] = m[band_cols].mean(axis=1)
    print(f"{s}: {m.shape[0]} genes x {m.shape[1]} periods, band {band_cols}, "
          f"{len(ref.index.intersection(m.index))} genes shared with reference, median score {scores[s].median():.1f}")

## Methods

- **baseline**: Pearson, all shared genes (what the pipeline's `correlate.py` does)
- **pearson_varN**: Pearson on the N genes whose expression varies most across groups (drops genes that are flat
  across cell types and can't distinguish them)
- **spearman_var5k**: Spearman on the top 5000 variable genes, for comparison only: rank ties on very sparse groups
  such as `erythrocyte_Blood` push them to the top

In [ ]:
def variable_genes(expr, n):
    return expr.var(axis=1).nlargest(n).index

def label(n):
    return f"pearson_var{n // 1000}k" if n >= 1000 and n % 1000 == 0 else f"pearson_var{n}"

def gene_set(method, expr):  # genes a method correlates over
    if method == "baseline":
        return expr.index
    n = 5000 if method.startswith("spearman") else int(method.split("var")[1].replace("k", "000"))
    return variable_genes(expr, n)

METHODS = ["baseline"] + [label(n) for n in VAR_GENE_COUNTS] + ["spearman_var5k"]

def correlate(method, expr, score):
    g = gene_set(method, expr)
    e, s = expr.loc[g], score.loc[g]
    if method.startswith("spearman"):
        return e.rank().corrwith(s.rank())
    return e.corrwith(s)

results = {}  # (method, sample) -> correlation per group
for s in SAMPLES:
    shared = ref.index.intersection(scores[s].index)
    for m in METHODS:
        results[(m, s)] = correlate(m, ref.loc[shared], scores[s].loc[shared]).sort_values()
print("methods:", METHODS)

## Method summary

In [ ]:
ERY = cat.index[cat == "erythroid"]
rows = []
for m in METHODS:
    for s in SAMPLES:
        r = results[(m, s)]
        rk = r.rank()
        rows.append({"method": m, "sample": s, "min_r": r.min(), "spread": r.max() - r.min(),
                     f"immune_top{TOP_N}": (cat[r.index[:TOP_N]] == "blood/immune").mean(),
                     "erythroid_median_rank": rk[ERY].median(),
                     "erythrocyte_Blood_rank": int(rk["erythrocyte_Blood"]), "top1": r.index[0]})
    rows[-1]["healthy_vs_cancer_rho"] = rows[-2]["healthy_vs_cancer_rho"] = \
        results[(m, "healthy")].corr(results[(m, "cancer")], method="spearman")
summary = pd.DataFrame(rows).set_index(["method", "sample"])
summary.to_csv(OUT_DIR / "method_summary.csv")
summary.round(3)

In [ ]:
# method comparison at a glance
n_groups = ref.shape[1]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
x = np.arange(len(METHODS)); w = 0.38
for i, s in enumerate(SAMPLES):
    sm = summary.xs(s, level="sample").loc[METHODS]
    off = (i - 0.5) * w
    axes[0].bar(x + off, sm["min_r"], w, color=SAMPLE_COLORS[s], label=s)
    axes[1].bar(x + off, sm["spread"], w, color=SAMPLE_COLORS[s])
    axes[2].bar(x + off, sm[f"immune_top{TOP_N}"], w, color=SAMPLE_COLORS[s])
    axes[3].bar(x + off, sm["erythroid_median_rank"], w, color=SAMPLE_COLORS[s])
rho = summary.xs("healthy", level="sample").loc[METHODS, "healthy_vs_cancer_rho"]
axes[1].plot(x, rho * summary["spread"].max(), "k.--", label="healthy vs cancer ρ (scaled)")
axes[3].axhline(n_groups, color="k", ls=":", lw=1); axes[3].axhline(n_groups / 2, color="k", ls=":", lw=0.6)
titles = ["strongest correlation (min r) - lower is stronger", "spread (max − min r) - higher separates more",
          f"fraction of top {TOP_N} that is blood/immune", f"erythroid median rank - higher is better (max {n_groups})"]
for ax, t in zip(axes, titles):
    ax.set_xticks(x); ax.set_xticklabels(METHODS, rotation=40, ha="right"); ax.set_title(t, fontsize=10)
axes[0].legend(); axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "method_comparison.png", dpi=150)
plt.show()

## Healthy vs cancer, per method

Each dot is a cell-type group: x = correlation in healthy, y = correlation in cancer; dashed line = equal in both.
A tight diagonal cloud = consistent method; the whole cloud off the diagonal = global (likely technical, e.g. depth)
shift; single groups far off the line = candidate sample-specific signal (needs replicates to be meaningful).

In [ ]:
cat_legend = [Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in CAT_COLORS.items()]
order = cat.sort_values(key=lambda c: c.map({"other": 0, "blood/immune": 1, "erythroid": 2})).index  # draw erythroid on top

ncol = 4
nrow = int(np.ceil(len(METHODS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.8 * nrow))
for ax, m in zip(axes.flat, METHODS):
    h, c = results[(m, "healthy")][order], results[(m, "cancer")][order]
    ax.scatter(h, c, s=np.where(cat[order] == "erythroid", 40, 10), c=cat_color[order], alpha=0.7)
    lo, hi = min(h.min(), c.min()), max(h.max(), c.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8)
    ax.set_title(f"{m}  (ρ = {h.corr(c, method='spearman'):.2f})")
    ax.set_xlabel("healthy r"); ax.set_ylabel("cancer r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=cat_legend, fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "scatter_healthy_vs_cancer.png", dpi=150)
plt.show()

## Correlations by cell-type category

Distribution of every group's correlation, split by category, for each method and sample. A useful method separates
blood/immune (more negative) from other groups, and keeps erythroid groups at or above the "other" level.

In [ ]:
cats = list(CAT_COLORS)
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 4.2 * nrow), sharey=False)
for ax, m in zip(axes.flat, METHODS):
    for i, k in enumerate(cats):
        for j, s in enumerate(SAMPLES):
            vals = results[(m, s)][cat[results[(m, s)].index] == k]
            pos = i * 3 + j
            bp = ax.boxplot(vals, positions=[pos], widths=0.8, patch_artist=True, showfliers=False)
            bp["boxes"][0].set(facecolor=SAMPLE_COLORS[s], alpha=0.5, edgecolor=CAT_COLORS[k])
            ax.scatter(np.random.default_rng(0).normal(pos, 0.08, len(vals)), vals, s=5, color=CAT_COLORS[k], alpha=0.5)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks([i * 3 + 0.5 for i in range(len(cats))]); ax.set_xticklabels(cats)
    ax.set_title(m); ax.set_ylabel("r")
for ax in list(axes.flat)[len(METHODS):]:
    ax.axis("off")
axes.flat[0].legend(handles=[Patch(color=c, alpha=0.5, label=s) for s, c in SAMPLE_COLORS.items()], fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / "category_distributions.png", dpi=150)
plt.show()

In [ ]:
# top groups per method as a table, erythroid groups marked with *
for s in SAMPLES:
    top = pd.DataFrame({m: [g + (" *" if cat[g] == "erythroid" else "") for g in results[(m, s)].index[:TOP_N]]
                        for m in METHODS})
    top.index = range(1, TOP_N + 1)
    print(f"=== {s}")
    display(top)

In [ ]:
allcorr = pd.DataFrame({f"{m}__{s}": r for (m, s), r in results.items()})
allcorr.insert(0, "category", cat)
allcorr.index.name = "cell_type"
allcorr.to_csv(OUT_DIR / "correlations_all_methods.csv")
print(f"wrote {OUT_DIR / 'correlations_all_methods.csv'}")

## Report figures: cell-type rankings

Rankings for the methods in `RANK_METHODS`. Groups are named `<cell type>_<tissue>`; the **cell-type level** figures
average each cell type's correlation over the tissues it was measured in (158 cell types instead of 447 groups),
which is easier to read. Colours as everywhere else; blue = healthy, orange = cancer.

With one healthy and one cancer sample, differences between the two are exploratory only.

In [ ]:
def split_group(g):
    cell_type, tissue = g.split("_", 1)
    return cell_type, tissue.replace("_", " ")

def by_celltype(r):
    # mean correlation per cell type over its tissues, most negative first
    ct = pd.Series([split_group(g)[0] for g in r.index], index=r.index)
    agg = r.groupby(ct).agg(["mean", "count"]).sort_values("mean")
    agg["category"] = [category(g) for g in r.groupby(ct).apply(lambda x: x.index[0]).loc[agg.index]]
    return agg

def bar_ranking(ax, values, labels, cats, title):
    y = np.arange(len(values))[::-1]
    ax.barh(y, values, color=[CAT_COLORS[c] for c in cats], alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels([f"{i + 1}. {l}" for i, l in enumerate(labels)], fontsize=8)
    for yi, v in zip(y, values):
        ax.text(v, yi, f"{v:.3f} ", va="center", ha="left" if v < 0 else "right", fontsize=7,
                color="white" if v < 0 else "black")
    ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("correlation (r)"); ax.set_title(title, fontsize=11)

cat_handles = [Patch(color=c, label=k) for k, c in CAT_COLORS.items()]
celltype_rank = {(m, s): by_celltype(results[(m, s)]) for m in RANK_METHODS for s in SAMPLES}

In [ ]:
# top cell types (averaged over tissues), one panel per sample
for m in RANK_METHODS:
    fig, axes = plt.subplots(1, len(SAMPLES), figsize=(7 * len(SAMPLES), 0.3 * RANK_TOP + 1.5))
    for ax, s in zip(axes, SAMPLES):
        top = celltype_rank[(m, s)].iloc[:RANK_TOP]
        labels = [f"{c}  [n={n}]" for c, n in zip(top.index, top["count"])]
        bar_ranking(ax, top["mean"].values, labels, top["category"], f"{s}: top {RANK_TOP} cell types")
    axes[-1].legend(handles=cat_handles, fontsize=8, loc="lower left")
    fig.suptitle(f"Cell-type ranking, {m} (mean r over tissues; n = number of tissues)", fontsize=12)
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"ranking_celltypes_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# biggest rank changes between healthy and cancer (cell-type level)
N_SHIFT = 15
for m in RANK_METHODS:
    rh = celltype_rank[(m, "healthy")]["mean"].rank()
    rc = celltype_rank[(m, "cancer")]["mean"].rank()
    shift = (rh - rc.loc[rh.index]).sort_values()   # positive = ranks higher (stronger) in cancer
    sel = pd.concat([shift.head(N_SHIFT), shift.tail(N_SHIFT)])
    fig, ax = plt.subplots(figsize=(8, 0.28 * len(sel) + 1.5))
    y = np.arange(len(sel))
    ax.barh(y, sel.values, color=[CAT_COLORS[celltype_rank[(m, "healthy")].loc[ct, "category"]] for ct in sel.index], alpha=0.85)
    ax.set_yticks(y); ax.set_yticklabels([f"{ct}  (#{int(rh[ct])} → #{int(rc[ct])})" for ct in sel.index], fontsize=8)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlabel(f"rank change (positive = stronger in cancer), out of {len(rh)} cell types")
    ax.set_title(f"Largest rank changes healthy → cancer ({m})\nexploratory: one sample per condition", fontsize=11)
    ax.legend(handles=cat_handles, fontsize=8, loc="lower right")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"rank_shift_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# top 10 cell types per sample and method, as a table
tbl = {}
for m in RANK_METHODS:
    for s in SAMPLES:
        top = celltype_rank[(m, s)].iloc[:10]
        tbl[(m, s)] = [f"{c} ({v:.3f})" for c, v in zip(top.index, top["mean"])]
top10 = pd.DataFrame(tbl, index=range(1, 11))
top10.to_csv(OUT_DIR / "top10_celltypes.csv")
top10

## Report figures: rank distributions (simplified versions of Stanley et al. 2024, Fig. 2)

Each group's correlation is turned into a **rank within the sample** (1 = most negative = strongest signal,
447 = weakest). The paper's boxplots are over many samples; with one sample per condition, the boxes here are
over the **groups inside a category** (all cell types of a compartment, all cell types in a tissue, all tissues of a
cell type), shown side by side for healthy (blue) and cancer (orange). Y axes are flipped so better ranks are higher.
Black × = erythroid groups (negative control).

In [ ]:
REPORT_METHOD = RANK_METHODS[0]
COMP_COLORS = {"immune": "tab:red", "epithelial": "tab:green", "endothelial": "tab:purple", "stromal": "tab:brown"}
ranks = {s: results[(REPORT_METHOD, s)].rank() for s in SAMPLES}   # 1 = most negative
n_groups = len(ranks[SAMPLES[0]])
tissue = pd.Series({g: split_group(g)[1] for g in ref.columns})
celltype = pd.Series({g: split_group(g)[0] for g in ref.columns})

def paired_boxes(ax, keys, members, show_ery=True, vertical=True):
    # members(key) -> list of groups; draws a healthy and a cancer box per key
    rng = np.random.default_rng(0)
    for i, k in enumerate(keys):
        groups = members(k)
        for j, s in enumerate(SAMPLES):
            pos = i + (j - 0.5) * 0.38
            vals = ranks[s][groups]
            bp = ax.boxplot(vals, positions=[pos], widths=0.34, patch_artist=True, showfliers=False, vert=vertical)
            bp["boxes"][0].set(facecolor=SAMPLE_COLORS[s], alpha=0.45)
            jit = rng.normal(pos, 0.05, len(vals))
            ery = np.array([cat[g] == "erythroid" for g in groups])
            xy = (jit, vals) if vertical else (vals, jit)
            ax.scatter(*(a[~ery] for a in xy), s=6, color="k", alpha=0.35, zorder=3)
            if show_ery and ery.any():
                ax.scatter(*(np.asarray(a)[ery] for a in xy), s=40, marker="x", color="black", zorder=4)
    sample_handles = [Patch(color=c, alpha=0.45, label=s) for s, c in SAMPLE_COLORS.items()]
    ery_handle = Line2D([], [], marker="x", ls="", color="black", label="erythroid")
    ax.legend(handles=sample_handles + [ery_handle], fontsize=8)

In [ ]:
# D. ranks by compartment
keys = list(COMP_COLORS)
fig, ax = plt.subplots(figsize=(8, 5))
paired_boxes(ax, keys, lambda k: list(comp.index[comp == k]))
ax.set_xticks(range(len(keys))); ax.set_xticklabels([f"{k}\n(n={int((comp == k).sum())})" for k in keys])
for lbl, k in zip(ax.get_xticklabels(), keys):
    lbl.set_color(COMP_COLORS[k])
ax.invert_yaxis(); ax.set_ylabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Correlation rank by compartment ({REPORT_METHOD})")
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_by_compartment_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# E. ranks by tissue, sorted by healthy median rank
keys = tissue.groupby(tissue).apply(lambda t: ranks["healthy"][t.index].median()).sort_values().index
fig, ax = plt.subplots(figsize=(16, 5.5))
paired_boxes(ax, keys, lambda k: list(tissue.index[tissue == k]))
ax.set_xticks(range(len(keys))); ax.set_xticklabels([f"{k} ({int((tissue == k).sum())})" for k in keys], rotation=45, ha="right")
ax.invert_yaxis(); ax.set_ylabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Correlation rank by tissue, all cell types in the tissue ({REPORT_METHOD}); (n) = cell types")
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_by_tissue_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# F. immune compartment, ranks by cell type (points = tissues), sorted by healthy median rank
imm = comp.index[comp == "immune"]
keys = celltype[imm].groupby(celltype[imm]).apply(lambda t: ranks["healthy"][t.index].median()).sort_values().index
fig, ax = plt.subplots(figsize=(8, 0.28 * len(keys) + 1.5))
paired_boxes(ax, keys, lambda k: list(imm[celltype[imm] == k]), vertical=False)
ax.set_yticks(range(len(keys))); ax.set_yticklabels([f"{k} ({int((celltype[imm] == k).sum())})" for k in keys], fontsize=7)
ax.invert_yaxis()   # strongest cell types at the top
ax.set_xlabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Immune cell types, rank across tissues\n({REPORT_METHOD}; (n) = tissues)", fontsize=11)
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_immune_celltypes_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# H. macrophage rank by tissue of residence (one value per tissue and sample)
mac = celltype.index[celltype == "macrophage"]
order = ranks["healthy"][mac].sort_values().index
fig, ax = plt.subplots(figsize=(7, 0.32 * len(order) + 1.5))
y = np.arange(len(order))
for s in SAMPLES:
    ax.scatter(ranks[s][order], y, color=SAMPLE_COLORS[s], s=45, label=s, zorder=3)
ax.hlines(y, ranks["healthy"][order], ranks["cancer"][order], color="lightgray", lw=2)
ax.set_yticks(y); ax.set_yticklabels([tissue[g] for g in order])
ax.invert_yaxis()
ax.set_xlim(0, n_groups); ax.set_xlabel(f"rank (1 = strongest, {n_groups} = weakest)")
ax.set_title(f"Macrophage rank by tissue of residence ({REPORT_METHOD})"); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT_DIR / f"ranks_macrophage_by_tissue_{REPORT_METHOD}.png", dpi=150)
plt.show()

In [ ]:
# top and bottom of the ranking (cell type + tissue), coloured by compartment, erythroid in black
for m in RANK_METHODS:
    fig, axes = plt.subplots(2, len(SAMPLES), figsize=(7.5 * len(SAMPLES), 0.3 * 2 * RANK_TOP + 2.5))
    for j, s in enumerate(SAMPLES):
        r = results[(m, s)]
        for i, (part, sel) in enumerate((("top", r.iloc[:RANK_TOP]), ("bottom", r.iloc[-RANK_TOP:]))):
            ax = axes[i, j]
            start = 1 if part == "top" else len(r) - RANK_TOP + 1
            y = np.arange(len(sel))[::-1]
            colors = ["black" if cat[g] == "erythroid" else COMP_COLORS[comp[g]] for g in sel.index]
            ax.barh(y, sel.values, color=colors, alpha=0.85)
            ax.set_yticks(y)
            ax.set_yticklabels([f"{start + k}. {ct} ({t})" for k, (ct, t) in enumerate(map(split_group, sel.index))], fontsize=7)
            ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("correlation (r)")
            ax.xaxis.set_major_locator(plt.MaxNLocator(5))
            ax.set_title(f"{s}: {part} {RANK_TOP} of {len(r)}", fontsize=10)
    handles = [Patch(color=c, label=k) for k, c in COMP_COLORS.items()] + [Patch(color="black", label="erythroid")]
    fig.suptitle(f"Strongest and weakest cell type / tissue groups ({m})", fontsize=12)
    fig.legend(handles=handles, fontsize=8, loc="upper right", ncol=len(handles))
    plt.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(OUT_DIR / f"ranking_top_bottom_{m}.png", dpi=150, bbox_inches="tight")
    plt.show()

## S curves

For every period column (120-280 bp), correlate each gene's FFT intensity at that period with each group's
expression, using the same gene set as the method. Lines are the mean over each category's groups (shaded = ±1 SD
across groups); solid = healthy, dashed = cancer. Contributing cell types should dip around 193-199 bp (gold band);
erythroid should stay flat.

In [ ]:
def period_curves(method, m):
    shared = ref.index.intersection(m.index)
    g = gene_set(method, ref.loc[shared])
    expr = ref.loc[g]
    return pd.DataFrame({p: expr.corrwith(m.loc[g, p]) for p in m.columns}).T  # periods x groups

curves = {(meth, s): period_curves(meth, fft[s]) for meth in CURVE_METHODS for s in SAMPLES}

def cat_mean(curve, k):
    sub = curve.loc[:, cat[curve.columns] == k]
    return sub.mean(axis=1), sub.std(axis=1)

def band(ax):
    ax.axvspan(*BAND, color="gold", alpha=0.25)
    ax.axhline(0, color="k", lw=0.5)

In [ ]:
# one panel per method: colour = category, solid = healthy, dashed = cancer
SAMPLE_STYLES = {"healthy": "-", "cancer": "--"}
fig, axes = plt.subplots(1, len(CURVE_METHODS), figsize=(8 * len(CURVE_METHODS), 5), squeeze=False)
for ax, meth in zip(axes[0], CURVE_METHODS):
    band(ax)
    for k, col in CAT_COLORS.items():
        for s in SAMPLES:
            mu, sd = cat_mean(curves[(meth, s)], k)
            ax.plot(mu.index, mu, color=col, lw=2, ls=SAMPLE_STYLES[s])
            ax.fill_between(mu.index, mu - sd, mu + sd, color=col, alpha=0.05)
    ax.set_title(meth); ax.set_xlabel("period (bp)"); ax.set_ylabel("mean r over groups")
handles = ([Line2D([], [], color=c, lw=2, label=k) for k, c in CAT_COLORS.items()]
           + [Line2D([], [], color="dimgray", lw=2, ls=ls, label=s) for s, ls in SAMPLE_STYLES.items()])
axes[0, -1].legend(handles=handles, fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
fig.savefig(OUT_DIR / "scurves.png", dpi=150)
plt.show()